In [7]:
import os
import json
import re
import pandas as pd
import numpy as np
import pprint

from utils import exact_match_results, relax_match_results


* model_folder: output folder without similarity algorithm
* model_sim_folder: output folder after running similarity algorithm

In [8]:
# model_folder = 'Output_gemma4_26b_no_sim'
model_folder = "Output_llama3.1_8b_no_sim"
# model_folder = "Output_mistral-small_24b_no_sim"
# model_folder = "Output_gemma4_31b_no_sim"

model_sim_folder = 'Output_llama3.1_8b_sim'
# model_sim_folder = "Output_gemma4_31b_sim"
# model_sim_folder = "Output_mistral-small_24b_sim"
# model_sim_folder = "Output_gemma4_26b_sim"

if os.path.exists(f"{model_sim_folder}") == False:
    os.makedirs(f"{model_sim_folder}")

if os.path.exists(f"{model_sim_folder}") == False:
    os.makedirs(f"{model_sim_folder}")

In [9]:
# # Checking if there's any null output:
# for sample in os.listdir(model_folder):
#     with open(f"{model_folder}/{sample}", "r") as f:
#         data = json.load(f)
#         if not data:
#             print(f"Sample {sample} has null output.")


# Identify all rephrased entities

In [10]:
preds_dict_od_all = []
# preds_dict_od_all_final = []
preds_dict_all_temp = []
for sample in os.listdir(model_folder):
    # if sample in ['sample_68', 'sample_635', 'sample_71', 'sample_324','sample_2748']:
        # print(f"evaluating {sample}:")
        with open(f'label/{sample}.json', 'r') as f:
            label_dict = json.load(f)

        #Read output
        with open(f"{model_folder}/{sample}", 'r') as f:
            preds_dict = json.load(f)
        # print(preds_dict)
        #Read original text
        with open(f'notes/{sample}.txt', 'r') as file:
            text = file.read()
        preds_dict_all_temp.append(preds_dict)
        preds_dict_processed = []
        preds_dict_od = []
        for ent in preds_dict:
            for phrase, label in ent.items():
        
                matches = list(re.finditer(re.escape(phrase), text,re.IGNORECASE))
                if matches:
                    for match in matches:
                        # print(f"Found '{match.group()}' at [{match.start()}, {match.end()}]")
                        preds_dict_processed.append({'entity':match.group(), 'label': label, 'start': match.start(), 'end': match.end()})
                else:

                    preds_dict_od.append({'entity':phrase, 'label': 'OD', 'start': 0, 'end': 0})
                    preds_dict_od_all.append({'sample':sample,'entity':phrase, 'label': 'OD', 'start': 0, 'end': 0})
        
        # for item in preds_dict_od_all:
        #     if item not in preds_dict_od_all_final:
        #         preds_dict_od_all_final.append(item)
        # with open(f"{model_attribution_folder}/{sample}", 'w') as f:
        #     json.dump(preds_dict_od, f, indent=2)

print(len(preds_dict_od_all))
pprint.pprint(preds_dict_od_all)

88
[{'end': 0,
  'entity': 'hearing problems',
  'label': 'OD',
  'sample': 'sample_134_20210112211711',
  'start': 0},
 {'end': 0,
  'entity': 'darkening of the eyes',
  'label': 'OD',
  'sample': 'sample_169_20210112211715',
  'start': 0},
 {'end': 0,
  'entity': 'yellowing of the urine',
  'label': 'OD',
  'sample': 'sample_169_20210112211715',
  'start': 0},
 {'end': 0,
  'entity': 'enlargement of the spleen',
  'label': 'OD',
  'sample': 'sample_169_20210112211715',
  'start': 0},
 {'end': 0,
  'entity': 'surgery',
  'label': 'OD',
  'sample': 'sample_169_20210112211715',
  'start': 0},
 {'end': 0,
  'entity': 'back pain',
  'label': 'OD',
  'sample': 'sample_46_20210610044418_Hayden',
  'start': 0},
 {'end': 0,
  'entity': 'buttock pain',
  'label': 'OD',
  'sample': 'sample_46_20210610044418_Hayden',
  'start': 0},
 {'end': 0,
  'entity': 'osteoporosis',
  'label': 'OD',
  'sample': 'sample_2789',
  'start': 0},
 {'end': 0,
  'entity': 'constipation',
  'label': 'OD',
  'sample'

# Similarity algorithm

### Original Method

In [18]:
from sentence_transformers import SentenceTransformer, util
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=device)
print(f"Using device: {device}")

for sample in os.listdir(model_folder):
    # if sample == 'sample_134_20210112211711':
    #Read original text
    #Read label dict
        print(f"processing: {sample}")

        with open(f'label/{sample}.json', 'r') as f:
            label_dict = json.load(f)
            
        #Read output
        with open(f"{model_folder}/{sample}", 'r') as f:
            org_preds_json = json.load(f)

        #Read original text
        with open(f'notes/{sample}.txt', 'r') as file:
            text = file.read()
        
        #Find the messed up tokens in the output
        preds_dict = []
        messup_pred_tokens = []
        for ent in org_preds_json:
            for phrase, label in ent.items():
                matches = list(re.finditer(re.escape(phrase), text, re.IGNORECASE))
                if matches:
                    for match in matches:
                        # print(f"Found '{match.group()}' at [{match.start()}, {match.end()}]")
                        preds_dict.append({'entity':match.group(), 'label': label, 'start': match.start(), 'end': match.end()})
                else:
                    messup_pred_tokens.append(phrase)
        
  
        #Clean up the note
        clean_text = re.sub(r'[.,:&]', '', text)
        clean_original_tokens = clean_text.split()
        n = len(clean_original_tokens)
        print(f"length of the text: {n}")

        
        # Attribution algorithm
        unique_messup_pred_tokens = list(dict.fromkeys(messup_pred_tokens))
        max_target_length = max((len(token.split()) for token in unique_messup_pred_tokens if token.strip()), default=0)
        target_lengths = range(1, max_target_length*5)
        
        spans = []
        seen_spans = set()
        for i in range(n):
            remaining = n - i
            for width in target_lengths:
                if width > remaining:
                    break
                span = ' '.join(clean_original_tokens[i:i + width])
                if span not in seen_spans:
                    seen_spans.add(span)
                    spans.append(span)

        if spans and unique_messup_pred_tokens:
            spans_emb = model.encode(
                spans,
                batch_size=128,
                convert_to_tensor=True,
                normalize_embeddings=True,
            
            ).to(device)

            target_emb = model.encode(
                unique_messup_pred_tokens,
                batch_size=128,
                convert_to_tensor=True,
                normalize_embeddings=True,

            ).to(device)

            sim = util.cos_sim(target_emb, spans_emb)
            top_indices = sim.argmax(dim=1).tolist()
            resolved_tokens = [spans[idx] for idx in top_indices]
            resolved_mapping = dict(zip(unique_messup_pred_tokens, resolved_tokens))
            alg_pred_tokens = [resolved_mapping[token] for token in messup_pred_tokens]
        else:
            alg_pred_tokens = []

        #Write back the output
        mapping = dict(zip(messup_pred_tokens, alg_pred_tokens))
        original = []
        updated = []

        for record in org_preds_json:
            original_record = {}
            updated_record = {}

            for key, value in record.items():

                if key in mapping:
                    original_record[key] = "false positive"
                    updated_record[mapping[key]] = value
                    
                else:
                    original_record[key] = value
                    updated_record[key] = value

            original.append(original_record)
            updated.append(updated_record)
        
        with open(f"{model_sim_folder}/{sample}", 'w') as f:
            json.dump(updated, f, indent=2)


Using device: cuda
processing: sample_134_20210112211711
length of the text: 305
2
processing: sample_25_20210610044410_Isra
length of the text: 372
0
processing: sample_38_20210610044413_Hayden
length of the text: 540
0
processing: sample_323
length of the text: 279
0
processing: sample_169_20210112211715
length of the text: 801
4
processing: sample_1246
length of the text: 336
0
processing: sample_46_20210610044418_Hayden
length of the text: 228
2
processing: sample_2789
length of the text: 767
1
processing: sample_116_20210112211709
length of the text: 210
0
processing: sample_1262
length of the text: 816
3
processing: sample_2764
length of the text: 415
0
processing: sample_76_20210112211704
length of the text: 241
0
processing: sample_37_20210621151709_Hayden
length of the text: 285
0
processing: sample_38_20210604143816_Isra
length of the text: 511
0
processing: sample_2757
length of the text: 394
0
processing: sample_80_20210604143825_Booma
length of the text: 233
0
processing: 

### New Method

In [19]:
def find_max_sim_containing_token(model, token_idx, token_sim, ai_output_encode, tokens, checked, max_span):
  l = token_idx
  r = token_idx + 1

  lb = l
  rb = r

  n = len(tokens)

  max_sim = token_sim
  txt = tokens[token_idx]
  max_sim_text = txt

  # check right-hand side
  while (r<n) and (not checked[r]):
    txt += f' {tokens[r]}'
    r += 1
    text_encode = model.encode(txt,
                              batch_size=128,
                              convert_to_tensor=True,
                              normalize_embeddings=True,
                               ).to(device)
    sim = util.cos_sim(text_encode, ai_output_encode).item()
    # print(f'{txt}: {sim}')
    if sim > max_sim:
      max_sim_text = txt
      max_sim = sim
      rb = r
    if r-(token_idx + 1) >= max_span:
      break

  txt = max_sim_text

  # check left-hand side
  while (l>0) and (not checked[l-1]):
    l -= 1
    txt = f'{tokens[l]} ' + txt
    text_encode = model.encode(txt,
                              batch_size=128,
                              convert_to_tensor=True,
                              normalize_embeddings=True,
                               ).to(device)
    sim = util.cos_sim(text_encode, ai_output_encode).item()
    # print(f'{txt}: {sim}')
    if sim > max_sim:
      max_sim_text = txt
      max_sim = sim
      lb = l
    if token_idx-l >= max_span:
      break

  return max_sim_text, max_sim

def find_max_sim(model, ai_output, tokens, max_span=1000):
  n = len(tokens)
  checked = [False] * n
  ai_output_encode = model.encode(ai_output,
                              batch_size=128,
                              convert_to_tensor=True,
                              normalize_embeddings=True,
                               ).to(device)

  sims = []
  for i, token in enumerate(tokens):
    text_encode = model.encode(token,
                              batch_size=128,
                              convert_to_tensor=True,
                              normalize_embeddings=True,
                               ).to(device)
    sim = util.cos_sim(text_encode, ai_output_encode).item()
    sims.append((i, sim))

  sims.sort(key=lambda x: x[1], reverse=True)
  # print(sims)
  # print("---------------------------------------")

  best_match = ""
  best_score = float("-inf")

  for i in sims:
    y_max = 0
    x, y = find_max_sim_containing_token(model, i[0], i[1], ai_output_encode, tokens, checked, max_span)
    checked[i[0]] = True
    if y > best_score:
      best_score = y
      best_match = x
    # print(f"Starting from '{tokens[i[0]]}': {x}: {y}")
    # print("---------------------------------------")
  return best_match

from sentence_transformers import SentenceTransformer, util
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=device)
print(f"Using device: {device}")

for sample in os.listdir(model_folder):
    # if sample == 'sample_134_20210112211711':
      print(f"processing: {sample}")

      with open(f'label/{sample}.json', 'r') as f:
          label_dict = json.load(f)
          
      #Read output
      with open(f"{model_folder}/{sample}", 'r') as f:
          org_preds_json = json.load(f)

      #Read original text
      with open(f'notes/{sample}.txt', 'r') as file:
          text = file.read()

      #Find the messed up tokens in the output
      preds_dict = []
      messup_pred_tokens = []
      for ent in org_preds_json:
          for phrase, label in ent.items():
              matches = list(re.finditer(re.escape(phrase), text, re.IGNORECASE))
              if matches:
                  for match in matches:
                      # print(f"Found '{match.group()}' at [{match.start()}, {match.end()}]")
                      preds_dict.append({'entity':match.group(), 'label': label, 'start': match.start(), 'end': match.end()})
              else:
                  messup_pred_tokens.append(phrase)
    
      
      # Clean up the note
      clean_text = re.sub(r'[.,:&]', '', text)
      clean_original_tokens = clean_text.split()
      n = len(clean_original_tokens)
      print(f"length of the text: {n}")

      #Attribution algorithm
      unique_messup_pred_tokens = list(dict.fromkeys(messup_pred_tokens))
      resolved_mapping = {}

      for ai_output in unique_messup_pred_tokens:
          best_match = find_max_sim(
              model,
              ai_output,
              clean_original_tokens,
              max_span=(len(ai_output.split()) * 5 - 1))
          
          resolved_mapping[ai_output] = best_match

      alg_pred_tokens = [resolved_mapping[token] for token in messup_pred_tokens]

      #Write back the output
      mapping = dict(zip(messup_pred_tokens, alg_pred_tokens))
      original = []
      updated = []

      for record in org_preds_json:
          original_record = {}
          updated_record = {}

          for key, value in record.items():

              if key in mapping:
                  original_record[key] = "false positive"
                  updated_record[mapping[key]] = value
                  
              else:
                  original_record[key] = value
                  updated_record[key] = value

          original.append(original_record)
          updated.append(updated_record)
      
      with open(f"{model_sim_folder}/{sample}", 'w') as f:
          json.dump(updated, f, indent=2)




Using device: cuda
processing: sample_134_20210112211711
length of the text: 305
processing: sample_25_20210610044410_Isra
length of the text: 372
processing: sample_38_20210610044413_Hayden
length of the text: 540
processing: sample_323
length of the text: 279
processing: sample_169_20210112211715
length of the text: 801
processing: sample_1246
length of the text: 336
processing: sample_46_20210610044418_Hayden
length of the text: 228
processing: sample_2789
length of the text: 767
processing: sample_116_20210112211709
length of the text: 210
processing: sample_1262
length of the text: 816
processing: sample_2764
length of the text: 415
processing: sample_76_20210112211704
length of the text: 241
processing: sample_37_20210621151709_Hayden
length of the text: 285
processing: sample_38_20210604143816_Isra
length of the text: 511
processing: sample_2757
length of the text: 394
processing: sample_80_20210604143825_Booma
length of the text: 233
processing: sample_367
length of the text: 5

# Evaluation - Attribution Results

In [20]:
tp_rm, fp_rm, fn_rm, od_rm = 0, 0, 0, 0
tp_em, fp_em, fn_em, od_em = 0, 0, 0, 0
for sample in os.listdir(model_sim_folder):
    # if sample not in ['sample_140_20210112211712']:
        print(f"evaluating {sample}:")
        with open(f'label/{sample}.json', 'r') as f:
            label_dict = json.load(f)

        #Read output
        with open(f"{model_sim_folder}/{sample}", 'r') as f:
            preds_dict = json.load(f)
        # print(preds_dict)
        #Read original text
        with open(f'notes/{sample}.txt', 'r') as file:
            text = file.read()

        preds_dict_processed = []
        for ent in preds_dict:
            for phrase, label in ent.items():
        
                matches = list(re.finditer(re.escape(phrase), text,re.IGNORECASE))
                if matches:
                    for match in matches:
                        # print(f"Found '{match.group()}' at [{match.start()}, {match.end()}]")
                        preds_dict_processed.append({'entity':match.group(), 'label': label, 'start': match.start(), 'end': match.end()})
                else:
                    preds_dict_processed.append({'entity':phrase, 'label': 'OD', 'start': 0, 'end': 0})
                
        #Deduplication
        preds_dict_final = []
        seen = set()
        for d in preds_dict_processed:
            t = tuple(sorted(d.items()))
            if t not in seen:
                seen.add(t)
                preds_dict_final.append(d)
                
        ## Evaluation
        em, em_mm, em_ud, em_od = exact_match_results(label_dict, preds_dict_final, 'Output_verification', sample)
        rm, rm_mm, rm_ud, rm_od = relax_match_results(label_dict, preds_dict_final, 'Output_verification', sample)
    
        tp_rm+= rm
        fp_rm+= rm_mm
        fn_rm+= rm_ud
        od_rm+= rm_od

            
        tp_em+= em
        fp_em+= em_mm
        fn_em+= em_ud
        od_em+= em_od

rm_precision = tp_rm/(tp_rm + rm_mm + od_rm)
rm_recall = tp_rm/(tp_rm + fn_rm)

em_precision = tp_em/(tp_em + em_mm + od_em)
em_recall = tp_em/(tp_em + fn_em)

print(f"SUMMARY EXACT MATCH: TP {tp_em}, FP {fp_em}, UD {fn_em}, OD {od_em}")
print(f"MICRO AVERAGE SCORE EXACT MATCH: Precision: {em_precision}, Recall: {em_recall}, F1 score: {2*em_precision*em_recall/(em_precision + em_recall)}")

print(f"SUMMARY RELAX MATCH: TP {tp_rm}, FP {fp_rm}, UD {fn_rm}, OD {od_rm}")
print(f"MICRO AVERAGE SCORE RELAX MATCH: Precision: {rm_precision}, Recall: {rm_recall}, F1 score: {2*rm_precision*rm_recall/(rm_precision + rm_recall)}")

evaluating sample_134_20210112211711:
[{'entity': 'General Medicine', 'label': 'TREATMENT', 'start': 33, 'end': 49}, {'entity': 'general medicine', 'label': 'TREATMENT', 'start': 2267, 'end': 2283}, {'entity': 'Normal Female ROS Template', 'label': 'TEST', 'start': 63, 'end': 89}, {'entity': 'Normal female review of systems template', 'label': 'TEST', 'start': 103, 'end': 143}, {'entity': 'balance problems', 'label': 'PROBLEM', 'start': 467, 'end': 483}, {'entity': 'PULMONARY', 'label': 'TEST', 'start': 978, 'end': 987}, {'entity': 'bleeding', 'label': 'PROBLEM', 'start': 611, 'end': 619}, {'entity': 'bleeding', 'label': 'PROBLEM', 'start': 1380, 'end': 1388}, {'entity': 'pain', 'label': 'PROBLEM', 'start': 1190, 'end': 1194}, {'entity': 'pain', 'label': 'PROBLEM', 'start': 1633, 'end': 1637}, {'entity': 'pain', 'label': 'PROBLEM', 'start': 1645, 'end': 1649}, {'entity': 'UTI', 'label': 'PROBLEM', 'start': 245, 'end': 248}, {'entity': 'cholesterol', 'label': 'PROBLEM', 'start': 1981, '

[{'entity': 'disseminated intravascular coagulation', 'label': 'PROBLEM', 'start': 2571, 'end': 2609}, {'entity': 'sepsis', 'label': 'PROBLEM', 'start': 2615, 'end': 2621}, {'entity': 'DIAGNOSES', 'label': 'TEST', 'start': 351, 'end': 360}, {'entity': '2', 'label': 'TREATMENT', 'start': 405, 'end': 406}, {'entity': '2', 'label': 'TREATMENT', 'start': 574, 'end': 575}, {'entity': '2', 'label': 'TREATMENT', 'start': 1137, 'end': 1138}, {'entity': '2', 'label': 'TREATMENT', 'start': 2468, 'end': 2469}, {'entity': '2', 'label': 'TREATMENT', 'start': 2512, 'end': 2513}, {'entity': '2', 'label': 'TREATMENT', 'start': 2963, 'end': 2964}, {'entity': 'clinically', 'label': 'PROBLEM', 'start': 1112, 'end': 1122}, {'entity': 'factor replacement', 'label': 'TREATMENT', 'start': 2787, 'end': 2805}, {'entity': 'surgical history', 'label': 'PROBLEM', 'start': 1386, 'end': 1402}, {'entity': 'illicit drugs', 'label': 'PROBLEM', 'start': 1440, 'end': 1453}, {'entity': 'family', 'label': 'PROBLEM', 'star

# Evaluation - Results before attribution

In [ ]:
tp_rm, fp_rm, fn_rm, od_rm = 0, 0, 0, 0
tp_em, fp_em, fn_em, od_em = 0, 0, 0, 0
for sample in os.listdir(model_folder):
    # if sample not in ['sample_140_20210112211712']:
        print(f"evaluating {sample}:")
        with open(f'label/{sample}.json', 'r') as f:
            label_dict = json.load(f)

        #Read output
        with open(f"{model_folder}/{sample}", 'r') as f:
            preds_dict = json.load(f)
        # print(preds_dict)
        #Read original text
        with open(f'notes/{sample}.txt', 'r') as file:
            text = file.read()

        preds_dict_processed = []
        for ent in preds_dict:
            for phrase, label in ent.items():
        
                matches = list(re.finditer(re.escape(phrase), text,re.IGNORECASE))
                if matches:
                    for match in matches:
                        # print(f"Found '{match.group()}' at [{match.start()}, {match.end()}]")
                        preds_dict_processed.append({'entity':match.group(), 'label': label, 'start': match.start(), 'end': match.end()})
                else:
                    preds_dict_processed.append({'entity':phrase, 'label': 'OD', 'start': 0, 'end': 0})
                
        #Deduplication
        preds_dict_final = []
        seen = set()
        for d in preds_dict_processed:
            t = tuple(sorted(d.items()))
            if t not in seen:
                seen.add(t)
                preds_dict_final.append(d)
                
        ## Evaluation
        em, em_mm, em_ud, em_od = exact_match_results(label_dict, preds_dict_final, 'Output_verification', sample)
        rm, rm_mm, rm_ud, rm_od = relax_match_results(label_dict, preds_dict_final, 'Output_verification', sample)
    
        tp_rm+= rm
        fp_rm+= rm_mm
        fn_rm+= rm_ud
        od_rm+= rm_od

            
        tp_em+= em
        fp_em+= em_mm
        fn_em+= em_ud
        od_em+= em_od

rm_precision = tp_rm/(tp_rm + rm_mm + od_rm)
rm_recall = tp_rm/(tp_rm + fn_rm)

em_precision = tp_em/(tp_em + em_mm + od_em)
em_recall = tp_em/(tp_em + fn_em)

print(f"SUMMARY EXACT MATCH: TP {tp_em}, FP {fp_em}, UD {fn_em}, OD {od_em}")
print(f"MICRO AVERAGE SCORE EXACT MATCH: Precision: {em_precision}, Recall: {em_recall}, F1 score: {2*em_precision*em_recall/(em_precision + em_recall)}")

print(f"SUMMARY RELAX MATCH: TP {tp_rm}, FP {fp_rm}, UD {fn_rm}, OD {od_rm}")
print(f"MICRO AVERAGE SCORE RELAX MATCH: Precision: {rm_precision}, Recall: {rm_recall}, F1 score: {2*rm_precision*rm_recall/(rm_precision + rm_recall)}")

evaluating sample_134_20210112211711:
[{'entity': 'General Medicine', 'label': 'TREATMENT', 'start': 33, 'end': 49}, {'entity': 'general medicine', 'label': 'TREATMENT', 'start': 2267, 'end': 2283}, {'entity': 'Normal Female ROS Template', 'label': 'TEST', 'start': 63, 'end': 89}, {'entity': 'Normal female review of systems template', 'label': 'TEST', 'start': 103, 'end': 143}, {'entity': 'hearing problems', 'label': 'OD', 'start': 0, 'end': 0}, {'entity': 'balance problems', 'label': 'PROBLEM', 'start': 467, 'end': 483}, {'entity': 'PULMONARY', 'label': 'TEST', 'start': 978, 'end': 987}, {'entity': 'bleeding', 'label': 'PROBLEM', 'start': 611, 'end': 619}, {'entity': 'bleeding', 'label': 'PROBLEM', 'start': 1380, 'end': 1388}, {'entity': 'pain', 'label': 'PROBLEM', 'start': 1190, 'end': 1194}, {'entity': 'pain', 'label': 'PROBLEM', 'start': 1633, 'end': 1637}, {'entity': 'pain', 'label': 'PROBLEM', 'start': 1645, 'end': 1649}, {'entity': 'UTI', 'label': 'PROBLEM', 'start': 245, 'end':